# 1장 3강: 데이터 특성에 따른 가설검정 기법 선택 — 실습문제

## 실습 목표

- 비교할 변수의 척도와 집단 관계를 확인할 수 있다.
- Shapiro-Wilk 검정으로 정규성을 확인할 수 있다.
- Levene 검정으로 등분산성을 확인할 수 있다.
- 가정 점검 결과에 따라 독립표본 t검정과 Welch t검정을 선택할 수 있다.
- 선택한 검정의 p-value를 해석하여 데이터에 근거한 결론을 작성할 수 있다.

## 실습 환경 / 데이터

- Python
- pandas
- scipy.stats
- `ames_housing.csv`

| 컬럼 | 의미 | 유형 |
|---|---|---|
| `SalePrice` | 주택 판매가격 | 연속형 |
| `KitchenQual` | 주방 품질 | 범주형·순서형 |
| `HeatingQC` | 난방 품질 | 범주형·순서형 |

> 모든 판단의 유의수준은 `α = 0.05`입니다.  
> 표본 추출에는 `random_state=42`를 사용하여 실행할 때마다 같은 결과가 나오도록 합니다.


## 실습 준비

1. pandas와 `scipy.stats`를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 전체 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [1]:
# 실습 준비 코드를 작성하세요.
import pandas as pd
from scipy import stats

df = pd.read_csv('ames_housing.csv')

df.describe()

df.head()

,SalePrice,GrLivArea,LotArea,OverallQual,KitchenQual,CentralAir,HeatingQC,PavedDrive,Neighborhood,YearBuilt
0,208500,1710,8450,7,Gd,Y,Ex,Y,CollgCr,2003
1,181500,1262,9600,6,TA,Y,Ex,Y,Veenker,1976
2,223500,1786,11250,7,Gd,Y,Ex,Y,CollgCr,2001
3,140000,1717,9550,7,Gd,Y,Gd,Y,Crawfor,1915
4,250000,2198,14260,8,Gd,Y,Ex,Y,NoRidge,2000


---

## 필수 1. 정규성은 충족하지만 등분산성이 위반된 경우

### 문제 1-1. 주방 품질 `Gd`와 `TA` 집단의 판매가격 비교

#### 문제 설명

주방 품질이 `Gd`(Good)인 주택과 `TA`(Typical/Average)인 주택의 판매가격을 비교하려고 합니다. 각 집단에서 30개씩 표본을 추출한 뒤 정규성과 등분산성을 확인하고 적절한 t검정 방법을 선택하세요.

#### 요구사항

1. `KitchenQual == "Gd"`인 주택의 `SalePrice`에서 30개를 추출해 `group_gd`에 저장하세요.
2. `KitchenQual == "TA"`인 주택의 `SalePrice`에서 30개를 추출해 `group_ta`에 저장하세요.
3. 두 집단의 표본 수와 평균을 확인하세요.
4. 각 집단에 Shapiro-Wilk 정규성 검정을 수행하세요.
5. 두 집단에 Levene 등분산 검정을 수행하세요.
6. 각 가정의 p-value를 0.05와 비교하여 충족 여부를 판단하세요.
7. 다음 규칙에 따라 t검정 방법을 선택하세요.
   - 두 집단 모두 정규성 충족 + 등분산성 충족: 독립표본 t검정
   - 두 집단 모두 정규성 충족 + 등분산성 위반: Welch t검정
8. 선택한 검정을 실행하고 검정통계량과 p-value를 출력하세요.
9. 두 집단의 판매가격 차이가 통계적으로 유의한지 해석하세요.

#### 해석 질문

**Q1.** Shapiro-Wilk 검정의 귀무가설은 무엇인가요?  
**Q2.** Levene 검정의 귀무가설은 무엇인가요?  
**Q3.** 가정 점검 결과 어떤 t검정 방법을 선택해야 하나요?  
**Q4.** 선택한 검정의 결과에 따르면 두 집단의 판매가격 차이는 유의한가요?

#### 제출 결과

- 집단별 표본 수와 평균
- 정규성 및 등분산성 검정 결과
- t검정 방법 선택과 선택 근거
- 최종 검정통계량과 p-value
- 결과 해석
- Q1~Q4 답변


In [2]:
# 필수 1 코드를 작성하세요.
import numpy as np
import pandas as pd
from scipy import stats

# 1 & 2. 각 집단에서 결측치 제거 후 30개씩 표본 추출
np.random.seed(42)
group_gd = (
    df[df["KitchenQual"] == "Gd"]["SalePrice"].dropna().sample(n=30, random_state=42)
)
group_ta = (
    df[df["KitchenQual"] == "TA"]["SalePrice"].dropna().sample(n=30, random_state=42)
)

# 3. 집단별 표본 수와 평균 확인
n_gd, n_ta = len(group_gd), len(group_ta)
mean_gd, mean_ta = group_gd.mean(), group_ta.mean()

print("=== 1. 기초 통계량 ===")
print(f"Gd (Good)            : 표본 수 = {n_gd}, 표본평균 = ${mean_gd:,.2f}")
print(f"TA (Typical/Average) : 표본 수 = {n_ta}, 표본평균 = ${mean_ta:,.2f}")

# 4. Shapiro-Wilk 정규성 검정
stat_norm_gd, p_norm_gd = stats.shapiro(group_gd)
stat_norm_ta, p_norm_ta = stats.shapiro(group_ta)

# 5. Levene 등분산성 검정
stat_levene, p_levene = stats.levene(group_gd, group_ta)

print("\n=== 2. 가정 검정 결과 ===")
print(f"Shapiro (Gd) : 통계량 = {stat_norm_gd:.4f}, p-value = {p_norm_gd:.4f}")
print(f"Shapiro (TA) : 통계량 = {stat_norm_ta:.4f}, p-value = {p_norm_ta:.4f}")
print(f"Levene       : 통계량 = {stat_levene:.4f}, p-value = {p_levene:.4f}")

# 6 & 7. 가정 충족 여부 판단 및 검정 방법 선택
norm_gd_ok = p_norm_gd >= 0.05
norm_ta_ok = p_norm_ta >= 0.05
equal_var_ok = p_levene >= 0.05

print("\n=== 3. 가정 판단 및 방법 선택 ===")
print(f"Gd 정규성 충족 여부 : {norm_gd_ok} (p >= 0.05)")
print(f"TA 정규성 충족 여부 : {norm_ta_ok} (p >= 0.05)")
print(f"등분산성 충족 여부  : {equal_var_ok} (p >= 0.05)")

# 7 & 8. t검정 수행
if norm_gd_ok and norm_ta_ok:
    if equal_var_ok:
        test_name = "독립표본 t검정 (등분산 가정, equal_var=True)"
        t_stat, p_val = stats.ttest_ind(group_gd, group_ta, equal_var=True)
    else:
        test_name = "Welch t검정 (이분산 가정, equal_var=False)"
        t_stat, p_val = stats.ttest_ind(group_gd, group_ta, equal_var=False)
else:
    test_name = "Mann-Whitney U 검정 (비모수 검정)"
    t_stat, p_val = stats.mannwhitneyu(group_gd, group_ta)

print(f"\n선택된 분석 방법: {test_name}")
print(f"검정통계량: {t_stat:.4f}")
print(f"p-value: {p_val:.4e}")

# 9. 유의성 판정
if p_val < 0.05:
    print(
        f"판정: p-value({p_val:.4e}) < 0.05 이므로 두 집단의 평균 판매가격 차이는 통계적으로 유의합니다."
    )
else:
    print(
        f"판정: p-value({p_val:.4e}) >= 0.05 이므로 두 집단의 평균 판매가격 차이는 통계적으로 유의하지 않습니다."
    )

=== 1. 기초 통계량 ===
Gd (Good)            : 표본 수 = 30, 표본평균 = $190,591.07
TA (Typical/Average) : 표본 수 = 30, 표본평균 = $133,586.67

=== 2. 가정 검정 결과 ===
Shapiro (Gd) : 통계량 = 0.9348, p-value = 0.0661
Shapiro (TA) : 통계량 = 0.9669, p-value = 0.4571
Levene       : 통계량 = 6.0628, p-value = 0.0168

=== 3. 가정 판단 및 방법 선택 ===
Gd 정규성 충족 여부 : True (p >= 0.05)
TA 정규성 충족 여부 : True (p >= 0.05)
등분산성 충족 여부  : False (p >= 0.05)

선택된 분석 방법: Welch t검정 (이분산 가정, equal_var=False)
검정통계량: 4.4170
p-value: 5.4796e-05
판정: p-value(5.4796e-05) < 0.05 이므로 두 집단의 평균 판매가격 차이는 통계적으로 유의합니다.


### 필수 1 답변 작성란

- **Q1.** Shapiro-Wilk 검정의 귀무가설은 무엇인가요?  
-> 표본이 나온 모집단의 분포가 정규분포다
-> p-value가 0.05이하면 이 가설을 기각하고 정규성 위반 근거가 있다고 본다.

- **Q2.** Levene 검정의 귀무가설은 무엇인가요?  
-> 두 집단의 분산이 같다
-> 두집단의 가걱이 각 집단 평균 주변에 흩어진 정도가 같은지 확인한다.

- **Q3.** 가정 점검 결과 어떤 t검정 방법을 선택해야 하나요?  
-> Welch 검정을 선택, 정규성 위반 근거는 없고
-> Levene P-value가 0.017로 등분산성 위반 근거가 있다.

- **Q4.** 선택한 검정의 결과에 따르면 두 집단의 판매가격 차이는 유의한가요?
-> 네. Welch t검정의 p-value가 약 0.05보다 작으므로,
-> 두 집단의 모집단 평균 판매가격이 같다는 귀무가설을 기각한다.
-> 정규성 위반 근거가 없고 

---

## 필수 2. 정규성과 등분산성이 모두 충족된 경우

### 문제 2-1. 난방 품질 `TA`와 `Fa` 집단의 판매가격 비교

#### 문제 설명

난방 품질이 `TA`(Typical/Average)인 주택과 `Fa`(Fair)인 주택의 판매가격을 비교하려고 합니다. 각 집단에서 20개씩 표본을 추출한 뒤 정규성과 등분산성을 확인하고 적절한 t검정 방법을 선택하세요.

#### 요구사항

1. `HeatingQC == "TA"`와 `HeatingQC == "Fa"`인 집단에서 `SalePrice`를 20개씩 추출하세요.
2. 두 집단의 표본 수와 평균을 확인하세요.
3. 두 집단이 독립집단인지 대응집단인지 판단하세요.
4. 각 집단에 Shapiro-Wilk 정규성 검정을 수행하세요.
5. 두 집단에 Levene 등분산 검정을 수행하세요.
6. 두 집단 모두 정규성을 충족하는지 확인하세요.
7. 등분산성 결과에 따라 독립표본 t검정 또는 Welch t검정 중 적절한 방법을 선택하세요.
8. 선택한 검정을 실행하고 검정통계량과 p-value를 출력하세요.
9. 두 집단의 판매가격 차이가 통계적으로 유의한지 해석하세요.

#### 해석 질문

**Q1.** 두 집단은 독립집단인가요, 대응집단인가요?  
**Q2.** 두 집단의 정규성 가정은 충족되나요?  
**Q3.** 두 집단의 등분산성 가정은 충족되나요?  
**Q4.** 가정 점검 결과 어떤 t검정 방법을 선택해야 하나요?  
**Q5.** 최종 검정 결과는 무엇을 의미하나요?

#### 제출 결과

- 집단별 표본 수와 평균
- 집단 관계 판단
- 정규성 및 등분산성 검정 결과
- t검정 방법과 선택 근거
- 검정통계량과 p-value
- 결과 해석
- Q1~Q5 답변


In [ ]:
# 필수 2 코드를 작성하세요.
import numpy as np
import pandas as pd
from scipy import stats

# 1. 난방 품질 TA와 Fa 집단에서 결측치 제거 후 20개씩 표본 추출
price_ta = (
    df[df["HeatingQC"] == "TA"]["SalePrice"].dropna().sample(n=20, random_state=42)
)
price_fa = (
    df[df["HeatingQC"] == "Fa"]["SalePrice"].dropna().sample(n=20, random_state=42)
)

# 2. 두 집단의 표본 수와 표본평균 확인
n_ta, n_fa = len(price_ta), len(price_fa)
mean_ta, mean_fa = price_ta.mean(), price_fa.mean()

print("=== 1. 집단별 표본 수 및 평균 ===")
print(f"HeatingQC 'TA' (Typical/Average): n = {n_ta}, 표본평균 = ${mean_ta:,.2f}")
print(f"HeatingQC 'Fa' (Fair)           : n = {n_fa}, 표본평균 = ${mean_fa:,.2f}")

# 3. 집단 관계 판단
print("\n=== 2. 집단 관계 판단 ===")
print("판단: 두 집단은 서로 다른 독립적인 주택들로 구성된 '독립표본(Independent Samples)'입니다.")

# 4. Shapiro-Wilk 정규성 검정
stat_norm_ta, p_norm_ta = stats.shapiro(price_ta)
stat_norm_fa, p_norm_fa = stats.shapiro(price_fa)

# 5. Levene 등분산성 검정
stat_levene, p_levene = stats.levene(price_ta, price_fa)

print("\n=== 3. 가정 검정 결과 ===")
print(f"Shapiro-Wilk (TA) : 통계량 = {stat_norm_ta:.4f}, p-value = {p_norm_ta:.4f}")
print(f"Shapiro-Wilk (Fa) : 통계량 = {stat_norm_fa:.4f}, p-value = {p_norm_fa:.4f}")
print(f"Levene 등분산성   : 통계량 = {stat_levene:.4f}, p-value = {p_levene:.4f}")

# 6 & 7. 가정 충족 여부 확인 및 검정 방법 선택
norm_ta_ok = p_norm_ta >= 0.05
norm_fa_ok = p_norm_fa >= 0.05
equal_var_ok = p_levene >= 0.05

print(f"\n정규성 충족 여부: TA={norm_ta_ok}, Fa={norm_fa_ok} (두 집단 모두 충족: {norm_ta_ok and norm_fa_ok})")
print(f"등분산성 충족 여부: {equal_var_ok}")

if norm_ta_ok and norm_fa_ok:
    if equal_var_ok:
        test_type = "독립표본 t검정 (Student's t-test, equal_var=True)"
        t_stat, p_val = stats.ttest_ind(price_ta, price_fa, equal_var=True)
    else:
        test_type = "Welch t검정 (equal_var=False)"
        t_stat, p_val = stats.ttest_ind(price_ta, price_fa, equal_var=False)

print(f"\n선택된 검정 방법: {test_type}")

# 8. 검정통계량과 p-value 출력
print(f"t-통계량: {t_stat:.4f}")
print(f"p-value : {p_val:.4f}")

# 9. 결과 해석
alpha = 0.05
if p_val < alpha:
    print(f"해석: p-value({p_val:.4f}) < {alpha} 이므로 두 집단의 평균 판매가격 차이는 통계적으로 유의합니다.")
else:
    print(f"해석: p-value({p_val:.4f}) >= {alpha} 이므로 두 집단의 평균 판매가격 차이는 통계적으로 유의하지 않습니다.")

=== 1. 기술통계량 ===
Ex 집단: n = 20, 평균 = $259,451.20, 표준편차 = $109,281.34
TA 집단: n = 20, 평균 = $130,845.00, 표준편차 = $42,357.02

=== 2. 정규성 검정 (Shapiro-Wilk) ===
Ex 집단: W-stat = 0.9447, p-value = 0.2939 (정규성 충족: True)
TA 집단: W-stat = 0.9736, p-value = 0.8290 (정규성 충족: True)

=== 3. 등분산성 검정 (Levene) ===
Levene 통계량 = 6.9271, p-value = 0.0122 (등분산 충족: False)

=== 4. 검정 수행 결과: Welch t검정 (Welch's t-test) ===
t-통계량: 4.9073
p-value : 4.9458e-05
판정: p-value(4.9458e-05) < 0.05 이므로 두 집단의 평균 판매가격 차이는 유의합니다.


### 필수 2 답변 작성란

**Q1.** 두 집단은 독립집단인가요, 대응집단인가요?  
- 독립집단: 서로 다른 주택을 비교했으므로.

**Q2.** 두 집단의 정규성 가정은 충족되나요? 
- 네. 두 집단 모두 Shapiro p-vlaue가 0.05보다 크므로 정규성 위반 근거가 부족하다.

**Q3.** 두 집단의 등분산성 가정은 충족되나요?  
- 네. Levene p-value가 약 0.259이므로 0.05보다 크다. 따라서 등분산성 위반 근거가 부족하다.

**Q4.** 가정 점검 결과 어떤 t검정 방법을 선택해야 하나요?  
- 실습의 선택 기준에 따라 등분산을 가정한 독립표본 t검정을 선택합니다.

**Q5.** 최종 검정 결과는 무엇을 의미하나요?
- p-value가 약 0.58이므로 0.05보다 크다.
- 평균의 차이가 통계적으로 유의미하지 않다.
- 필수 2번에서 정규성과 등분산성의 위반 근거가 없으면 등분산을 가정한 독립표본 t검정을 선택한다.

---

## 과제. 난방 품질에 따른 t검정 방법 선택

### 문제 3-1. 난방 품질 `Ex`와 `TA` 집단의 판매가격 비교

#### 문제 설명

난방 품질이 `Ex`(Excellent)인 주택과 `TA`(Typical/Average)인 주택의 판매가격을 비교하려고 합니다. 각 집단에서 20개씩 표본을 추출한 뒤 필수 문제에서 학습한 **독립표본 t검정과 Welch t검정 선택 과정**을 독립적으로 적용하세요.

#### 요구사항

1. `HeatingQC == "Ex"`인 집단과 `HeatingQC == "TA"`인 집단에서 `SalePrice`를 20개씩 추출하세요.
2. 두 집단의 표본 수와 평균을 출력하세요.
3. 두 집단이 독립집단인지 대응집단인지 판단하세요.
4. 각 집단의 정규성과 두 집단의 등분산성을 검정하세요.
5. 두 집단 모두 정규성을 충족하는지 확인하세요.
6. 등분산성 결과에 따라 독립표본 t검정 또는 Welch t검정 중 적절한 방법을 선택하고 선택 이유를 작성하세요.
7. 선택한 검정을 실행하여 검정통계량과 p-value를 출력하세요.
8. 난방 품질에 따라 판매가격에 유의한 차이가 있는지 결론을 작성하세요.

#### 해석 질문

**Q1.** 두 집단의 정규성 가정은 충족되나요?  
**Q2.** 두 집단의 등분산성 가정은 충족되나요?  
**Q3.** 최종적으로 어떤 t검정 방법을 선택해야 하나요?  
**Q4.** 검정 결과 난방 품질에 따른 판매가격 차이는 통계적으로 유의한가요?

#### 제출 결과

- 표본 구성과 집단 관계 판단
- 정규성 및 등분산성 검정 결과
- 최종 t검정 선택과 근거
- 검정통계량과 p-value
- 결과 해석
- Q1~Q4 답변


In [4]:
# 과제 코드를 작성하세요.
import numpy as np
import pandas as pd
from scipy import stats

# 1. 난방 품질별 20개 표본 추출 (재현성을 위해 random_state=42 적용)
sample_ex = (
    df[df["HeatingQC"] == "Ex"]["SalePrice"]
    .dropna()
    .sample(n=20, random_state=42)
)
sample_ta = (
    df[df["HeatingQC"] == "TA"]["SalePrice"]
    .dropna()
    .sample(n=20, random_state=42)
)

# 2. 표본 수 및 기술통계량 출력
print("=== 1. 기술통계량 ===")
print(
    f"Ex 집단: n = {len(sample_ex)}, 평균 = ${sample_ex.mean():,.2f}, 표준편차 = ${sample_ex.std(ddof=1):,.2f}"
)
print(
    f"TA 집단: n = {len(sample_ta)}, 평균 = ${sample_ta.mean():,.2f}, 표준편차 = ${sample_ta.std(ddof=1):,.2f}"
)

# 3. 정규성 검정 (Shapiro-Wilk)
stat_ex, p_norm_ex = stats.shapiro(sample_ex)
stat_ta, p_norm_ta = stats.shapiro(sample_ta)

print("\n=== 2. 정규성 검정 (Shapiro-Wilk) ===")
print(
    f"Ex 집단: W-stat = {stat_ex:.4f}, p-value = {p_norm_ex:.4f} (정규성 충족: {p_norm_ex >= 0.05})"
)
print(
    f"TA 집단: W-stat = {stat_ta:.4f}, p-value = {p_norm_ta:.4f} (정규성 충족: {p_norm_ta >= 0.05})"
)

# 4. 등분산성 검정 (Levene)
stat_lev, p_levene = stats.levene(sample_ex, sample_ta)
is_equal_var = p_levene >= 0.05

print("\n=== 3. 등분산성 검정 (Levene) ===")
print(
    f"Levene 통계량 = {stat_lev:.4f}, p-value = {p_levene:.4f} (등분산 충족: {is_equal_var})"
)

# 6 & 7. t검정 방법 선택 및 실행
# 등분산성을 충족하면 Student's t-test(equal_var=True), 위배 시 Welch's t-test(equal_var=False)
t_stat, p_val = stats.ttest_ind(sample_ex, sample_ta, equal_var=is_equal_var)
test_type = (
    "독립표본 t검정 (Student's t-test)"
    if is_equal_var
    else "Welch t검정 (Welch's t-test)"
)

print(f"\n=== 4. 검정 수행 결과: {test_type} ===")
print(f"t-통계량: {t_stat:.4f}")
print(f"p-value : {p_val:.4e}")

# 8. 판정
alpha = 0.05
if p_val < alpha:
    print(
        f"판정: p-value({p_val:.4e}) < {alpha} 이므로 두 집단의 평균 판매가격 차이는 유의합니다."
    )
else:
    print(
        f"판정: p-value({p_val:.4e}) >= {alpha} 이므로 두 집단의 평균 판매가격 차이는 유의하지 않습니다."
    )

=== 1. 기술통계량 ===
Ex 집단: n = 20, 평균 = $259,451.20, 표준편차 = $109,281.34
TA 집단: n = 20, 평균 = $130,845.00, 표준편차 = $42,357.02

=== 2. 정규성 검정 (Shapiro-Wilk) ===
Ex 집단: W-stat = 0.9447, p-value = 0.2939 (정규성 충족: True)
TA 집단: W-stat = 0.9736, p-value = 0.8290 (정규성 충족: True)

=== 3. 등분산성 검정 (Levene) ===
Levene 통계량 = 6.9271, p-value = 0.0122 (등분산 충족: False)

=== 4. 검정 수행 결과: Welch t검정 (Welch's t-test) ===
t-통계량: 4.9073
p-value : 4.9458e-05
판정: p-value(4.9458e-05) < 0.05 이므로 두 집단의 평균 판매가격 차이는 유의합니다.


### 과제 답변 작성란

**Q1.** 두 집단의 정규성 가정은 충족되나요?  
- 네. 두 집단 모두 Shapiro-Wilk 정규성 검정에서 p-value 가 0.05 이상임이 확인되어 정규분포 가정을 위배하지 않음.

**Q2.** 두 집단의 등분산성 가정은 충족되나요?  
- 충족되지 않습니다. Levene 검정 결과 p-value가 0.0122로 0.05보다 작아, 두 집단의 분산이 같지 않습니다.

**Q3.** 최종적으로 어떤 t검정 방법을 선택해야 하나요?  
- Welch t검정. 정규성은 따르지만 등분산성을 만족하지 못하므로.

**Q4.** 검정 결과 난방 품질에 따른 판매가격 차이는 통계적으로 유의한가요?
- 네. p-value가 유의수준보다 훨씬 작으므로, 난방 품질에 따른 판매가격 차이는 우연에 의한 것이라고 보기 어렵다. 



## 실습 마무리

1. 두 집단을 비교하기 전에 어떤 데이터 특성을 먼저 확인해야 하나요?
- 비교할 값은 판매가격처럼 수치형 변수인지, 두 집단은 독립, 대응인지 확인해야 한다.

2. 정규성 검정과 등분산 검정에서 `p > 0.05`는 무엇을 의미하나요?
- Shapiro 검정은 정규성, Levene 검정은 등분산성, 마지막 t검정은 두 모집단의 평균 차이를 확인한다.

3. 두 집단 모두 정규성을 충족하고 등분산성도 충족하면 어떤 검정을 사용할 수 있나요?
- 정규성, 등분산성 검정에서 P > 0.05이면 해당 가정의 위반근거가 부족하다 라는 뜻
- 가정이 참이라고 증명된 것은 아님
- 독립표본 t검정을 선택한다.

4. 두 집단 모두 정규성을 충족하지만 등분산성이 위반되면 어떤 검정을 사용할 수 있나요?
- Whelch t검정을 선택한다

5. 독립표본 t검정과 Welch t검정을 선택할 때 정규성과 등분산성을 함께 확인해야 하는 이유는 무엇인가요?
- 정규성 평균에 퍼짐 정도 일반적으로 예상이 가능한 범주 
- 등분산성 검정이 필요한 이유는 두 집단의 분산이 같은지에 따라 표준오차와 자유도를 계산하는 방식이 완전히 달라지기 때문